# Notebook 15 - Portfolio Workflow Review and Case Study

Notebook 15 is the Milestone 18 portfolio workflow review notebook for `christophermoverton/fintech-stratlake-notebook-workflows`. It is a source-safe reviewer entry point for a native Fintech + StratLake workflow.

Repository purpose: this repository stages a source-safe notebook workflow around upstream Fintech market-data ingestion and upstream StratLake feature, strategy, portfolio, evidence, and governance command surfaces. Upstream apps own native runtime behavior; this notebook owns reviewer-facing orchestration, command preview, guardrails, and non-claim boundaries.

The notebook intentionally orchestrates native command surfaces rather than implementing notebook-owned ingestion, feature, strategy, backtest, portfolio, archive, restore, evidence-review, or governance logic.

Committed source must remain output-free, execution-count-null, preview-default, and free of secrets, runtime IDs, local paths, generated artifacts, logs, and executed outputs.


## 2. Workflow Thesis and Evidence Boundary

Notebook 15 demonstrates a portfolio workflow sequence: Fintech market-data ingestion or restore, session persistence handoff, StratLake feature generation or restore, feature validation, native strategy execution and backtest artifact review, native portfolio workflow execution through `stratlake-run-portfolio`, archive checkpoint and restore handoff, evidence review, governance observation, and promotion-evidence caveats.

Workflow map from Notebook 00-14: Notebook 00 establishes setup and storage boundaries; Notebooks 01-03 cover Fintech ingestion, session persistence, backup, and restore; Notebooks 04-06 stage StratLake session, feature generation, validation, archive, and handoff; Notebooks 07-12 cover strategy and backtest research review; Notebook 13 covers native campaign execution guardrails; Notebook 14 covers evidence review and governance observation.

It does not demonstrate investment quality, alpha, approval, promotion readiness, governance readiness, production readiness, deployment readiness, live-trading suitability, or source/runtime equivalence.


## 3. Runtime Profile Selector

The committed default is `portfolio_preview`. Runtime profiles describe workflow intent, but each native action also requires `NOTEBOOK15_ALLOW_NATIVE_COMMAND_EXECUTION=1` and its own explicit `NOTEBOOK15_ALLOW_...` gate.

Runtime override examples must remain commented in committed source.

```python
# import os
# os.environ["NOTEBOOK15_PROFILE"] = "portfolio_execution_run"
# os.environ["NOTEBOOK15_ALLOW_NATIVE_COMMAND_EXECUTION"] = "1"
# os.environ["NOTEBOOK15_ALLOW_PORTFOLIO_EXECUTION"] = "1"
# os.environ["NOTEBOOK15_MARK_PORTFOLIO_INPUTS_USER_REVIEWED"] = "true"
# os.environ["NOTEBOOK15_PORTFOLIO_CONFIG"] = "/content/stratlake-trade-engine-demo/configs/portfolios.yml"
# os.environ["NOTEBOOK15_PORTFOLIO_NAME"] = "momentum_meanrev_equal"
```


In [ ]:
import json
import os
import shlex
import shutil
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = None

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def env_true(name: str, default: bool = False) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "on"}


def command_text(command: list[str]) -> str:
    return " ".join(shlex.quote(str(part)) for part in command)


def display_rows(rows: list[dict[str, Any]], *, max_rows: int = 30) -> None:
    shown = rows[:max_rows]
    if pd is not None and display is not None:
        display(pd.DataFrame(shown))
    else:
        for row in shown:
            print(row)


def inspect_path(path: Path, classification: str) -> dict[str, Any]:
    return {
        "path": path.as_posix(),
        "exists": path.exists(),
        "is_dir": path.is_dir(),
        "classification": classification,
    }


CAVEATS: list[str] = []
COMMAND_RESULTS: list[dict[str, Any]] = []
CHECKPOINT_RESULTS: list[dict[str, Any]] = []


In [ ]:
NOTEBOOK15_PROFILE = os.environ.get("NOTEBOOK15_PROFILE", "portfolio_preview").strip() or "portfolio_preview"
NOTEBOOK15_PROFILE = {"portfolio_archive_restore": "portfolio_archive_restore_review"}.get(NOTEBOOK15_PROFILE, NOTEBOOK15_PROFILE)

VALID_NOTEBOOK15_PROFILES = {
    "portfolio_preview",
    "fintech_market_data_ingestion_run",
    "fintech_market_data_archive_restore",
    "stratlake_feature_generation_run",
    "stratlake_feature_archive_restore",
    "strategy_execution_run",
    "strategy_artifact_restore_review",
    "portfolio_execution_run",
    "portfolio_archive_restore_review",
    "workflow_archive_checkpoint",
    "workflow_archive_restore",
}
if NOTEBOOK15_PROFILE not in VALID_NOTEBOOK15_PROFILES:
    raise ValueError(f"Unsupported NOTEBOOK15_PROFILE: {NOTEBOOK15_PROFILE!r}")

ALLOW_NATIVE_COMMAND_EXECUTION = os.environ.get("NOTEBOOK15_ALLOW_NATIVE_COMMAND_EXECUTION", "0").strip() == "1"
RUN_STRATEGIES_AFTER_RESTORE = env_true("NOTEBOOK15_RUN_STRATEGIES_AFTER_RESTORE", default=False)
RUN_PORTFOLIO_AFTER_RESTORE = env_true("NOTEBOOK15_RUN_PORTFOLIO_AFTER_RESTORE", default=False)
ARCHIVE_AFTER_PROFILE_RUN = env_true("NOTEBOOK15_ARCHIVE_AFTER_PROFILE_RUN", default=False)

PROFILE_REQUEST_INSTALL = NOTEBOOK15_PROFILE != "portfolio_preview"
PROFILE_REQUEST_DRIVE = NOTEBOOK15_PROFILE != "portfolio_preview"
PROFILE_REQUEST_FINTECH_INIT = NOTEBOOK15_PROFILE in {"fintech_market_data_ingestion_run", "fintech_market_data_archive_restore", "stratlake_feature_generation_run"}
PROFILE_REQUEST_FINTECH_INGESTION = NOTEBOOK15_PROFILE == "fintech_market_data_ingestion_run"
PROFILE_REQUEST_FINTECH_ARCHIVE_RESTORE = NOTEBOOK15_PROFILE in {"fintech_market_data_archive_restore", "workflow_archive_restore", "portfolio_archive_restore_review"}
PROFILE_REQUEST_STRATLAKE_INIT = NOTEBOOK15_PROFILE in {"fintech_market_data_ingestion_run", "stratlake_feature_generation_run", "stratlake_feature_archive_restore", "strategy_execution_run", "portfolio_execution_run"}
PROFILE_REQUEST_FEATURE_GENERATION = NOTEBOOK15_PROFILE in {"fintech_market_data_ingestion_run", "stratlake_feature_generation_run"}
PROFILE_REQUEST_FEATURE_ARCHIVE_RESTORE = NOTEBOOK15_PROFILE in {"stratlake_feature_archive_restore", "workflow_archive_restore", "portfolio_archive_restore_review"}
PROFILE_REQUEST_STRATEGY_EXECUTION = NOTEBOOK15_PROFILE in {"fintech_market_data_ingestion_run", "strategy_execution_run"} or (NOTEBOOK15_PROFILE in {"stratlake_feature_archive_restore", "strategy_artifact_restore_review", "portfolio_archive_restore_review", "workflow_archive_restore"} and RUN_STRATEGIES_AFTER_RESTORE)
PROFILE_REQUEST_STRATEGY_ARTIFACT_RESTORE = NOTEBOOK15_PROFILE in {"strategy_artifact_restore_review", "portfolio_archive_restore_review", "workflow_archive_restore"}
PROFILE_REQUEST_PORTFOLIO_EXECUTION = NOTEBOOK15_PROFILE in {"fintech_market_data_ingestion_run", "portfolio_execution_run"} or (NOTEBOOK15_PROFILE in {"strategy_artifact_restore_review", "portfolio_archive_restore_review", "workflow_archive_restore"} and RUN_PORTFOLIO_AFTER_RESTORE)
PROFILE_REQUEST_PORTFOLIO_ARCHIVE_RESTORE = NOTEBOOK15_PROFILE in {"portfolio_archive_restore_review", "workflow_archive_restore"}
PROFILE_REQUEST_WORKFLOW_ARCHIVE_CHECKPOINT = NOTEBOOK15_PROFILE in {"fintech_market_data_ingestion_run", "workflow_archive_checkpoint"} or (ARCHIVE_AFTER_PROFILE_RUN and NOTEBOOK15_PROFILE != "portfolio_preview")
PROFILE_REQUEST_WORKFLOW_ARCHIVE_RESTORE = NOTEBOOK15_PROFILE == "workflow_archive_restore"

ALLOW_INSTALL = PROFILE_REQUEST_INSTALL and env_true("NOTEBOOK15_ALLOW_PACKAGE_INSTALL")
ALLOW_DRIVE_MOUNT = PROFILE_REQUEST_DRIVE and env_true("NOTEBOOK15_ALLOW_DRIVE_MOUNT")
ALLOW_FINTECH_INIT = PROFILE_REQUEST_FINTECH_INIT and env_true("NOTEBOOK15_ALLOW_FINTECH_INIT")
ALLOW_FINTECH_INGESTION = PROFILE_REQUEST_FINTECH_INGESTION and env_true("NOTEBOOK15_ALLOW_FINTECH_INGESTION")
ALLOW_FINTECH_ARCHIVE_RESTORE = PROFILE_REQUEST_FINTECH_ARCHIVE_RESTORE and env_true("NOTEBOOK15_ALLOW_FINTECH_ARCHIVE_RESTORE")
ALLOW_STRATLAKE_INIT = PROFILE_REQUEST_STRATLAKE_INIT and env_true("NOTEBOOK15_ALLOW_STRATLAKE_INIT")
ALLOW_FEATURE_GENERATION = PROFILE_REQUEST_FEATURE_GENERATION and env_true("NOTEBOOK15_ALLOW_FEATURE_GENERATION")
ALLOW_FEATURE_ARCHIVE_RESTORE = PROFILE_REQUEST_FEATURE_ARCHIVE_RESTORE and env_true("NOTEBOOK15_ALLOW_FEATURE_ARCHIVE_RESTORE")
ALLOW_STRATEGY_EXECUTION = PROFILE_REQUEST_STRATEGY_EXECUTION and env_true("NOTEBOOK15_ALLOW_STRATEGY_EXECUTION")
ALLOW_STRATEGY_ARTIFACT_RESTORE = PROFILE_REQUEST_STRATEGY_ARTIFACT_RESTORE and env_true("NOTEBOOK15_ALLOW_STRATEGY_ARTIFACT_RESTORE")
ALLOW_PORTFOLIO_EXECUTION = PROFILE_REQUEST_PORTFOLIO_EXECUTION and env_true("NOTEBOOK15_ALLOW_PORTFOLIO_EXECUTION")
ALLOW_PORTFOLIO_ARCHIVE_RESTORE = PROFILE_REQUEST_PORTFOLIO_ARCHIVE_RESTORE and env_true("NOTEBOOK15_ALLOW_PORTFOLIO_ARCHIVE_RESTORE")
ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT = PROFILE_REQUEST_WORKFLOW_ARCHIVE_CHECKPOINT and env_true("NOTEBOOK15_ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT")
ALLOW_WORKFLOW_ARCHIVE_RESTORE = PROFILE_REQUEST_WORKFLOW_ARCHIVE_RESTORE and env_true("NOTEBOOK15_ALLOW_WORKFLOW_ARCHIVE_RESTORE")

GATE_VALUES = {
    "ALLOW_FINTECH_INGESTION": ALLOW_FINTECH_INGESTION,
    "ALLOW_FINTECH_ARCHIVE_RESTORE": ALLOW_FINTECH_ARCHIVE_RESTORE,
    "ALLOW_FEATURE_GENERATION": ALLOW_FEATURE_GENERATION,
    "ALLOW_FEATURE_ARCHIVE_RESTORE": ALLOW_FEATURE_ARCHIVE_RESTORE,
    "ALLOW_STRATEGY_EXECUTION": ALLOW_STRATEGY_EXECUTION,
    "ALLOW_STRATEGY_ARTIFACT_RESTORE": ALLOW_STRATEGY_ARTIFACT_RESTORE,
    "ALLOW_PORTFOLIO_EXECUTION": ALLOW_PORTFOLIO_EXECUTION,
    "ALLOW_PORTFOLIO_ARCHIVE_RESTORE": ALLOW_PORTFOLIO_ARCHIVE_RESTORE,
    "ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT": ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT,
    "ALLOW_WORKFLOW_ARCHIVE_RESTORE": ALLOW_WORKFLOW_ARCHIVE_RESTORE,
}
actual_enabled_gates = {name for name, value in GATE_VALUES.items() if value}

if ALLOW_FINTECH_INGESTION and ALLOW_FINTECH_ARCHIVE_RESTORE:
    raise RuntimeError("Fintech ingestion and Fintech archive restore cannot both be enabled.")
if ALLOW_FEATURE_GENERATION and ALLOW_FEATURE_ARCHIVE_RESTORE:
    raise RuntimeError("Feature generation and feature archive restore cannot both be enabled.")

profile_summary = {
    "selected_profile": NOTEBOOK15_PROFILE,
    "source_safe_committed_default": NOTEBOOK15_PROFILE == "portfolio_preview",
    "profile_requested_gates": sorted(name for name, value in {
        "PROFILE_REQUEST_FINTECH_INGESTION": PROFILE_REQUEST_FINTECH_INGESTION,
        "PROFILE_REQUEST_FINTECH_ARCHIVE_RESTORE": PROFILE_REQUEST_FINTECH_ARCHIVE_RESTORE,
        "PROFILE_REQUEST_FEATURE_GENERATION": PROFILE_REQUEST_FEATURE_GENERATION,
        "PROFILE_REQUEST_FEATURE_ARCHIVE_RESTORE": PROFILE_REQUEST_FEATURE_ARCHIVE_RESTORE,
        "PROFILE_REQUEST_STRATEGY_EXECUTION": PROFILE_REQUEST_STRATEGY_EXECUTION,
        "PROFILE_REQUEST_PORTFOLIO_EXECUTION": PROFILE_REQUEST_PORTFOLIO_EXECUTION,
        "PROFILE_REQUEST_WORKFLOW_ARCHIVE_CHECKPOINT": PROFILE_REQUEST_WORKFLOW_ARCHIVE_CHECKPOINT,
        "PROFILE_REQUEST_WORKFLOW_ARCHIVE_RESTORE": PROFILE_REQUEST_WORKFLOW_ARCHIVE_RESTORE,
    }.items() if value),
    "enabled_profile_gates": sorted(actual_enabled_gates),
    "native_command_execution_confirmed": ALLOW_NATIVE_COMMAND_EXECUTION,
}
print("Notebook 15 profile gate check passed.")
profile_summary


## 4. Workspace and Command Helpers

Commands are recorded as skipped previews unless the profile branch, the action-specific `NOTEBOOK15_ALLOW_...` gate, and `NOTEBOOK15_ALLOW_NATIVE_COMMAND_EXECUTION=1` are all active.


In [ ]:
WORKSPACE_ROOT = Path("/content") if IN_COLAB else Path.cwd()
DRIVE_FOLDER_NAME = os.environ.get("NOTEBOOK15_DRIVE_FOLDER_NAME", "REPLACE_WITH_DRIVE_FOLDER_NAME").strip()
DRIVE_ROOT = Path("<drive-root-placeholder>") if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME" else Path("/content") / "drive" / "MyDrive" / DRIVE_FOLDER_NAME
FINTECH_ROOT = WORKSPACE_ROOT / "fintech-market-ingestion-demo"
STRATLAKE_ROOT = WORKSPACE_ROOT / "stratlake-trade-engine-demo"
MARKETLAKE_ROOT = FINTECH_ROOT / "data" / "curated"
DAILY_BARS_ROOT = MARKETLAKE_ROOT / "bars_daily"
FEATURES_DAILY_ROOT = STRATLAKE_ROOT / "data" / "curated" / "features_daily"
STRATEGY_ARTIFACT_ROOT = STRATLAKE_ROOT / "artifacts" / "strategies"
PORTFOLIO_ARTIFACT_ROOT = STRATLAKE_ROOT / "artifacts" / "portfolios"
FINTECH_SESSION_NAME = "notebook15_fintech_portfolio_handoff"
STRATLAKE_SESSION_NAME = "notebook15_stratlake_portfolio_workflow"

if ALLOW_DRIVE_MOUNT and ALLOW_NATIVE_COMMAND_EXECUTION:
    if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
        raise ValueError("Set NOTEBOOK15_DRIVE_FOLDER_NAME before mounting or writing Drive-backed state.")
    if IN_COLAB and drive is not None:
        drive.mount("/content/drive")
elif ALLOW_DRIVE_MOUNT:
    CAVEATS.append("Drive mount not performed; native execution is not confirmed.")

ALLOW_WORKSPACE_INITIALIZATION = ALLOW_NATIVE_COMMAND_EXECUTION and (ALLOW_FINTECH_INIT or ALLOW_STRATLAKE_INIT or ALLOW_FINTECH_INGESTION or ALLOW_FEATURE_GENERATION)
if ALLOW_WORKSPACE_INITIALIZATION:
    FINTECH_ROOT.mkdir(parents=True, exist_ok=True)
    STRATLAKE_ROOT.mkdir(parents=True, exist_ok=True)

def run_command(command: list[str], *, cwd: Path | None = None, allow_run: bool = False, label: str = "") -> dict[str, Any]:
    profile_gate_active = bool(allow_run)
    should_execute = profile_gate_active and ALLOW_NATIVE_COMMAND_EXECUTION
    row = {
        "label": label or command[0],
        "command": command_text(command),
        "cwd": cwd.as_posix() if cwd else None,
        "profile_gate_active": profile_gate_active,
        "native_command_execution_confirmed": ALLOW_NATIVE_COMMAND_EXECUTION,
        "allow_run": should_execute,
        "skipped": not should_execute,
        "skipped_preview": profile_gate_active and not ALLOW_NATIVE_COMMAND_EXECUTION,
        "returncode": None,
        "stdout_tail": "",
        "stderr_tail": "",
        "started_at": utc_now_iso(),
    }
    if not should_execute:
        COMMAND_RESULTS.append(row)
        return row
    started = time.time()
    completed = subprocess.run(command, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    row.update({"returncode": completed.returncode, "stdout_tail": completed.stdout[-2000:], "stderr_tail": completed.stderr[-2000:], "elapsed_seconds": round(time.time() - started, 3), "finished_at": utc_now_iso()})
    COMMAND_RESULTS.append(row)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed for {label or command[0]}: {completed.stderr[-1000:]}")
    return row

def verify_checkpoint(label: str, path: Path, *, required_when: bool, classification: str, must_contain_any: list[str] | None = None) -> dict[str, Any]:
    exists = path.exists()
    matched_children: list[str] = []
    if exists and path.is_dir() and must_contain_any:
        for pattern in must_contain_any:
            matched_children.extend(sorted(child.name for child in path.glob(pattern))[:5])
    satisfied = exists and (not must_contain_any or bool(matched_children))
    row = {"checkpoint": label, "path": path.as_posix(), "required_by_profile": required_when, "exists": exists, "is_dir": path.is_dir(), "contains_expected_children": None if not must_contain_any else bool(matched_children), "matched_children_sample": matched_children[:5], "satisfied": bool(satisfied), "classification": classification}
    CHECKPOINT_RESULTS.append(row)
    if required_when and ALLOW_NATIVE_COMMAND_EXECUTION and not satisfied:
        raise RuntimeError(f"Required Notebook 15 checkpoint was not satisfied: {label} at {path}")
    if required_when and not satisfied:
        CAVEATS.append(f"Checkpoint {label!r} is required by profile {NOTEBOOK15_PROFILE!r} but was not observed; cold-smoke may continue only because native command execution is not confirmed.")
    return row

display_rows([
    {"name": "WORKSPACE_ROOT", "value": WORKSPACE_ROOT.as_posix(), "classification": "active_runtime_workspace"},
    {"name": "STRATLAKE_ROOT", "value": STRATLAKE_ROOT.as_posix(), "classification": "runtime_workspace_not_committed"},
    {"name": "NATIVE_EXECUTION_CONFIRMED", "value": str(ALLOW_NATIVE_COMMAND_EXECUTION), "classification": "second_level_runtime_confirmation"},
])


## 5. Fintech and StratLake Native Setup

Fintech initialization uses `fintech-init-project --root ... --notebooks --with-session --session-name ...`. Fintech ingestion uses `fintech-backfill-daily` with a reviewed symbols file, date range, daily-bars output root, feed, session source, and window. StratLake features are built with `stratlake-build-features --timeframe ... --start ... --end ... --tickers ... --marketlake-root ...`.


In [ ]:
NOTEBOOK15_SYMBOLS = os.environ.get("NOTEBOOK15_SYMBOLS", "SPY,QQQ,IWM,DIA").strip()
MARKET_DATA_START_DATE = os.environ.get("NOTEBOOK15_INGESTION_START", "2025-01-01").strip()
MARKET_DATA_END_DATE = os.environ.get("NOTEBOOK15_INGESTION_END", "2025-04-01").strip()
FEATURE_START = os.environ.get("NOTEBOOK15_FEATURE_START", MARKET_DATA_START_DATE).strip()
FEATURE_END = os.environ.get("NOTEBOOK15_FEATURE_END", MARKET_DATA_END_DATE).strip()
FEATURE_TIMEFRAME = os.environ.get("NOTEBOOK15_FEATURE_TIMEFRAME", "1D").strip() or "1D"
ALPACA_FEED = os.environ.get("ALPACA_FEED", "iex").strip() or "iex"
FINTECH_SESSION_ID = os.environ.get("NOTEBOOK15_FINTECH_SESSION_ID", FINTECH_SESSION_NAME).strip()
FINTECH_TICKERS_FILE = FINTECH_ROOT / "configs" / "notebook15_symbols.txt"
STRATLAKE_TICKERS_FILE = STRATLAKE_ROOT / "configs" / "notebook15_feature_symbols.txt"
reviewed_symbols = [symbol.strip().upper() for symbol in NOTEBOOK15_SYMBOLS.split(",") if symbol.strip()]

FINTECH_BACKUP_ROOT = Path(os.environ.get("NOTEBOOK15_FINTECH_BACKUP_ROOT", "<reviewed-fintech-backup-root>"))
FINTECH_BACKUP_ID_RAW = os.environ.get("NOTEBOOK15_FINTECH_BACKUP_ID", "notebook15-portfolio-workflow").strip()
FINTECH_BACKUP_ID = "".join(char if char.isalnum() or char in {"-", "_", "."} else "-" for char in FINTECH_BACKUP_ID_RAW).strip("-._") or "notebook15-portfolio-workflow"
FINTECH_BACKUP_PACK_DIR = FINTECH_BACKUP_ROOT / FINTECH_BACKUP_ID
FINTECH_BACKUP_COLLISION_POLICY = os.environ.get("NOTEBOOK15_FINTECH_BACKUP_COLLISION_POLICY", "reuse_existing").strip() or "reuse_existing"
if FINTECH_BACKUP_COLLISION_POLICY not in {"reuse_existing", "new_id", "fail"}:
    raise ValueError("NOTEBOOK15_FINTECH_BACKUP_COLLISION_POLICY must be one of: reuse_existing, new_id, fail")
if FINTECH_BACKUP_ID != FINTECH_BACKUP_ID_RAW:
    CAVEATS.append("Fintech backup ID was sanitized to the allowed backup-id character set before command construction.")

if (ALLOW_FINTECH_INGESTION or ALLOW_FEATURE_GENERATION) and ALLOW_NATIVE_COMMAND_EXECUTION:
    FINTECH_TICKERS_FILE.parent.mkdir(parents=True, exist_ok=True)
    STRATLAKE_TICKERS_FILE.parent.mkdir(parents=True, exist_ok=True)
    FINTECH_TICKERS_FILE.write_text("\n".join(reviewed_symbols) + "\n", encoding="utf-8")
    STRATLAKE_TICKERS_FILE.write_text("\n".join(reviewed_symbols) + "\n", encoding="utf-8")
elif ALLOW_FINTECH_INGESTION or ALLOW_FEATURE_GENERATION:
    CAVEATS.append("Ticker-file materialization skipped because native command execution is not confirmed.")

fintech_init_cmd = ["fintech-init-project", "--root", FINTECH_ROOT.as_posix(), "--notebooks", "--with-session", "--session-name", FINTECH_SESSION_NAME]
fintech_ingestion_cmd = ["fintech-backfill-daily", "--symbols", FINTECH_TICKERS_FILE.as_posix(), "--start", MARKET_DATA_START_DATE, "--end", MARKET_DATA_END_DATE, "--out", DAILY_BARS_ROOT.as_posix(), "--feed", ALPACA_FEED, "--source", f"session_{FINTECH_SESSION_ID}", "--window", os.environ.get("NOTEBOOK15_BACKFILL_WINDOW", "month")]
fintech_restore_cmd = ["fintech-backup-data", "restore", "--backup-pack-dir", FINTECH_BACKUP_PACK_DIR.as_posix(), "--restore-root", MARKETLAKE_ROOT.as_posix(), "--overwrite-policy", os.environ.get("NOTEBOOK15_FINTECH_RESTORE_OVERWRITE_POLICY", "fail")]
if fintech_restore_cmd[-1] not in {"fail", "replace", "merge"}:
    raise ValueError("NOTEBOOK15_FINTECH_RESTORE_OVERWRITE_POLICY must use Fintech vocabulary: fail, replace, or merge")
stratlake_init_cmd = ["stratlake-init-session", "--root", STRATLAKE_ROOT.as_posix(), "--project-name", STRATLAKE_SESSION_NAME, "--marketlake-root", MARKETLAKE_ROOT.as_posix(), "--drive-root", DRIVE_ROOT.as_posix(), "--enable-drive-persistence", "--notebook-configs"]
feature_generation_cmd = ["stratlake-build-features", "--timeframe", FEATURE_TIMEFRAME, "--start", FEATURE_START, "--end", FEATURE_END, "--tickers", STRATLAKE_TICKERS_FILE.as_posix(), "--marketlake-root", MARKETLAKE_ROOT.as_posix()]
STRATLAKE_ARCHIVE_DRIVE_ROOT = Path(os.environ.get("NOTEBOOK15_STRATLAKE_ARCHIVE_DRIVE_ROOT", "<reviewed-stratlake-archive-drive-root>"))
STRATLAKE_ARCHIVE_ID = os.environ.get("NOTEBOOK15_STRATLAKE_ARCHIVE_ID", "<reviewed-stratlake-archive-id>").strip() or "<reviewed-stratlake-archive-id>"
STRATLAKE_ARCHIVE_DRIVE_PACK_DIR = STRATLAKE_ARCHIVE_DRIVE_ROOT / STRATLAKE_ARCHIVE_ID
feature_restore_cmd = ["stratlake-session-archive-restore-bootstrap", "--archive-root", STRATLAKE_ARCHIVE_DRIVE_PACK_DIR.as_posix(), "--target-root", STRATLAKE_ROOT.as_posix(), "--validate-before-restore", "--inspect-before-restore", "--overwrite-policy", os.environ.get("NOTEBOOK15_STRATLAKE_RESTORE_OVERWRITE_POLICY", "overwrite_allowed")]

fintech_init_result = run_command(fintech_init_cmd, allow_run=ALLOW_FINTECH_INIT, label="fintech init project")
fintech_ingestion_result = run_command(fintech_ingestion_cmd, allow_run=ALLOW_FINTECH_INGESTION, label="fintech daily bars ingestion")
fintech_restore_result = run_command(fintech_restore_cmd, allow_run=ALLOW_FINTECH_ARCHIVE_RESTORE, label="fintech market-data archive restore")
stratlake_init_result = run_command(stratlake_init_cmd, allow_run=ALLOW_STRATLAKE_INIT, label="stratlake init session")
feature_generation_result = run_command(feature_generation_cmd, cwd=STRATLAKE_ROOT, allow_run=ALLOW_FEATURE_GENERATION, label="stratlake feature generation")
feature_restore_result = run_command(feature_restore_cmd, cwd=STRATLAKE_ROOT, allow_run=ALLOW_FEATURE_ARCHIVE_RESTORE, label="stratlake feature archive restore")

market_data_checkpoint = verify_checkpoint("fintech_curated_market_data_available", DAILY_BARS_ROOT, required_when=ALLOW_FINTECH_INGESTION or ALLOW_FINTECH_ARCHIVE_RESTORE or ALLOW_FEATURE_GENERATION, classification="required_market_data_handoff", must_contain_any=["*.parquet", "*.csv", "**/*.parquet", "**/*.csv"])
feature_checkpoint = verify_checkpoint("stratlake_features_available_for_strategy_execution", FEATURES_DAILY_ROOT, required_when=ALLOW_FEATURE_GENERATION or ALLOW_FEATURE_ARCHIVE_RESTORE or ALLOW_STRATEGY_EXECUTION or ALLOW_PORTFOLIO_EXECUTION, classification="required_feature_data_for_strategies_and_portfolio", must_contain_any=["*.parquet", "*.csv", "**/*.parquet", "**/*.csv"])
display_rows([market_data_checkpoint, feature_checkpoint])


## 6. Native Strategy Execution

Notebook 15 runs three or more supported native strategies through `stratlake-run-strategy`. The default set avoids `breakout` because `breakout` is native-supported but requires feature columns `high` and `low`; Notebook 15 fails explicit incompatible selections before native execution instead of fabricating missing inputs.


In [ ]:
DEFAULT_NOTEBOOK15_STRATEGIES = "momentum_v1,mean_reversion_v1,buy_and_hold_v1"
SELECTED_STRATEGIES = [item.strip() for item in os.environ.get("NOTEBOOK15_SELECTED_STRATEGIES", DEFAULT_NOTEBOOK15_STRATEGIES).split(",") if item.strip()]
SUPPORTED_STRATLAKE_STRATEGIES = {"breakout", "buy_and_hold_v1", "cross_section_momentum", "mean_reversion", "mean_reversion_v1", "mean_reversion_v1_safe_2026_q1", "momentum_v1", "pairs_trading", "residual_momentum", "seeded_random_v1", "sma_crossover_v1", "time_series_momentum", "volatility_regime_momentum", "weighted_cross_section_ensemble"}
STRATEGY_REQUIRED_INPUT_COLUMNS = {"breakout": {"high", "low"}}
unsupported_strategies = [strategy for strategy in SELECTED_STRATEGIES if strategy not in SUPPORTED_STRATLAKE_STRATEGIES]
if unsupported_strategies:
    raise ValueError(f"Unsupported Notebook 15 strategy selection: {unsupported_strategies}")
if len(SELECTED_STRATEGIES) < 3:
    raise ValueError("Notebook 15 requires at least three selected strategies for the portfolio case study.")

def discover_feature_columns(feature_root: Path, *, max_files: int = 8) -> set[str]:
    if not feature_root.exists():
        return set()
    candidate_files: list[Path] = []
    for pattern in ("*.parquet", "**/*.parquet", "*.csv", "**/*.csv"):
        candidate_files.extend(sorted(feature_root.glob(pattern)))
        if len(candidate_files) >= max_files:
            break
    columns: set[str] = set()
    for path in candidate_files[:max_files]:
        try:
            if path.suffix.lower() == ".parquet" and pd is not None:
                columns.update(str(name) for name in pd.read_parquet(path).columns)
            elif path.suffix.lower() == ".csv" and pd is not None:
                columns.update(str(name) for name in pd.read_csv(path, nrows=0).columns)
        except Exception as exc:
            CAVEATS.append(f"Feature-column preflight could not inspect {path.as_posix()}: {type(exc).__name__}: {exc}")
    return columns

observed_feature_columns = discover_feature_columns(FEATURES_DAILY_ROOT) if ALLOW_STRATEGY_EXECUTION and ALLOW_NATIVE_COMMAND_EXECUTION else set()
incompatible_strategy_inputs = []
for strategy in SELECTED_STRATEGIES:
    required_columns = STRATEGY_REQUIRED_INPUT_COLUMNS.get(strategy, set())
    missing_columns = sorted(required_columns - observed_feature_columns) if required_columns else []
    if missing_columns:
        incompatible_strategy_inputs.append({"strategy": strategy, "missing_input_columns": missing_columns})
if incompatible_strategy_inputs:
    raise RuntimeError("Notebook 15 strategy input preflight failed before native execution: " + repr(incompatible_strategy_inputs))

STRATEGIES_CONFIG = STRATLAKE_ROOT / "configs" / "strategies.yml"
STRATEGY_EXECUTION_DELAY = os.environ.get("NOTEBOOK15_STRATEGY_EXECUTION_DELAY", "1")
STRATEGY_TRANSACTION_COST_BPS = os.environ.get("NOTEBOOK15_TRANSACTION_COST_BPS", "1.0")
STRATEGY_SLIPPAGE_BPS = os.environ.get("NOTEBOOK15_SLIPPAGE_BPS", "0.5")
strategy_commands = [["stratlake-run-strategy", "--strategies-config", STRATEGIES_CONFIG.as_posix(), "--strategy", strategy, "--start", FEATURE_START, "--end", FEATURE_END, "--execution-delay", STRATEGY_EXECUTION_DELAY, "--transaction-cost-bps", STRATEGY_TRANSACTION_COST_BPS, "--slippage-bps", STRATEGY_SLIPPAGE_BPS] for strategy in SELECTED_STRATEGIES]
strategy_config_checkpoint = verify_checkpoint("strategy_config_available_for_native_execution", STRATEGIES_CONFIG, required_when=ALLOW_STRATEGY_EXECUTION, classification="required_native_strategy_catalog")
strategy_feature_checkpoint = verify_checkpoint("feature_root_available_before_strategy_execution", FEATURES_DAILY_ROOT, required_when=ALLOW_STRATEGY_EXECUTION, classification="required_feature_data_for_native_strategy_execution", must_contain_any=["*.parquet", "*.csv", "**/*.parquet", "**/*.csv"])
strategy_results = [run_command(command, cwd=STRATLAKE_ROOT, allow_run=ALLOW_STRATEGY_EXECUTION, label=f"strategy {strategy}") for strategy, command in zip(SELECTED_STRATEGIES, strategy_commands)]
strategy_artifact_review_root = Path(os.environ.get("NOTEBOOK15_STRATEGY_ARTIFACT_ROOT", STRATEGY_ARTIFACT_ROOT.as_posix()))
strategy_checkpoint = verify_checkpoint("strategy_artifacts_available_for_portfolio_execution", strategy_artifact_review_root, required_when=ALLOW_STRATEGY_EXECUTION or ALLOW_STRATEGY_ARTIFACT_RESTORE or ALLOW_PORTFOLIO_EXECUTION, classification="required_strategy_artifacts_for_portfolio_workflow", must_contain_any=["*.json", "*.parquet", "*.csv", "**/*.json", "**/*.parquet", "**/*.csv"])
display_rows([{"strategy": strategy, "execution_enabled": ALLOW_STRATEGY_EXECUTION, "command": command_text(command)} for strategy, command in zip(SELECTED_STRATEGIES, strategy_commands)] + [strategy_checkpoint])


## 7. Native Portfolio Command

Notebook 15 uses the Milestone 11 native portfolio CLI, `stratlake-run-portfolio`, not `stratlake-run-research-campaign`. The portfolio command is downstream of completed strategy artifacts and can use registry-backed component selection or explicit run IDs.

Baseline shape:

```bash
stratlake-run-portfolio \
  --portfolio-config configs/portfolios.yml \
  --portfolio-name momentum_meanrev_equal \
  --from-registry \
  --timeframe 1D
```

Supported argument groups include portfolio config/name, `--run-ids`, `--from-registry`, `--from-sweep-top-ranked`, `--from-candidate-selection`, `--evaluation`, `--output-dir`, `--timeframe`, optimizer override, risk overrides, volatility targeting, execution-friction controls, `--strict`, `--simulation`, and `--promotion-gates`.


In [ ]:
PORTFOLIO_CONFIG_PATH = Path(os.environ.get("NOTEBOOK15_PORTFOLIO_CONFIG", (STRATLAKE_ROOT / "configs" / "portfolios.yml").as_posix()))
PORTFOLIO_NAME = os.environ.get("NOTEBOOK15_PORTFOLIO_NAME", "momentum_meanrev_equal").strip()
PORTFOLIO_TIMEFRAME = os.environ.get("NOTEBOOK15_PORTFOLIO_TIMEFRAME", FEATURE_TIMEFRAME).strip() or "1D"
PORTFOLIO_OUTPUT_DIR = Path(os.environ.get("NOTEBOOK15_PORTFOLIO_OUTPUT_DIR", PORTFOLIO_ARTIFACT_ROOT.as_posix()))
PORTFOLIO_FROM_REGISTRY = env_true("NOTEBOOK15_PORTFOLIO_FROM_REGISTRY", default=True)
PORTFOLIO_RUN_IDS = [item.strip() for item in os.environ.get("NOTEBOOK15_PORTFOLIO_RUN_IDS", "").replace(",", " ").split() if item.strip()]
if PORTFOLIO_FROM_REGISTRY and PORTFOLIO_RUN_IDS:
    raise ValueError("Use either NOTEBOOK15_PORTFOLIO_FROM_REGISTRY=true or NOTEBOOK15_PORTFOLIO_RUN_IDS, not both.")

portfolio_execution_cmd = ["stratlake-run-portfolio", "--portfolio-config", PORTFOLIO_CONFIG_PATH.as_posix(), "--portfolio-name", PORTFOLIO_NAME, "--timeframe", PORTFOLIO_TIMEFRAME, "--output-dir", PORTFOLIO_OUTPUT_DIR.as_posix()]
if PORTFOLIO_FROM_REGISTRY:
    portfolio_execution_cmd.append("--from-registry")
elif PORTFOLIO_RUN_IDS:
    portfolio_execution_cmd.extend(["--run-ids", ",".join(PORTFOLIO_RUN_IDS)])

optional_portfolio_arg_pairs = [
    ("--evaluation", os.environ.get("NOTEBOOK15_PORTFOLIO_EVALUATION_CONFIG", "").strip()),
    ("--simulation", os.environ.get("NOTEBOOK15_PORTFOLIO_SIMULATION_CONFIG", "").strip()),
    ("--promotion-gates", os.environ.get("NOTEBOOK15_PORTFOLIO_PROMOTION_GATES", "").strip()),
    ("--optimizer-method", os.environ.get("NOTEBOOK15_PORTFOLIO_OPTIMIZER_METHOD", "").strip()),
    ("--volatility-target-volatility", os.environ.get("NOTEBOOK15_VOLATILITY_TARGET_VOLATILITY", "").strip()),
    ("--volatility-target-lookback", os.environ.get("NOTEBOOK15_VOLATILITY_TARGET_LOOKBACK", "").strip()),
    ("--risk-target-volatility", os.environ.get("NOTEBOOK15_RISK_TARGET_VOLATILITY", "").strip()),
    ("--risk-volatility-window", os.environ.get("NOTEBOOK15_RISK_VOLATILITY_WINDOW", "").strip()),
    ("--risk-var-confidence-level", os.environ.get("NOTEBOOK15_RISK_VAR_CONFIDENCE_LEVEL", "").strip()),
    ("--risk-cvar-confidence-level", os.environ.get("NOTEBOOK15_RISK_CVAR_CONFIDENCE_LEVEL", "").strip()),
    ("--risk-max-volatility-scale", os.environ.get("NOTEBOOK15_RISK_MAX_VOLATILITY_SCALE", "").strip()),
    ("--execution-delay", os.environ.get("NOTEBOOK15_PORTFOLIO_EXECUTION_DELAY", "").strip()),
    ("--transaction-cost-bps", os.environ.get("NOTEBOOK15_PORTFOLIO_TRANSACTION_COST_BPS", "").strip()),
    ("--slippage-bps", os.environ.get("NOTEBOOK15_PORTFOLIO_SLIPPAGE_BPS", "").strip()),
    ("--fixed-fee", os.environ.get("NOTEBOOK15_PORTFOLIO_FIXED_FEE", "").strip()),
    ("--slippage-model", os.environ.get("NOTEBOOK15_PORTFOLIO_SLIPPAGE_MODEL", "").strip()),
    ("--slippage-turnover-scale", os.environ.get("NOTEBOOK15_PORTFOLIO_SLIPPAGE_TURNOVER_SCALE", "").strip()),
    ("--slippage-volatility-scale", os.environ.get("NOTEBOOK15_PORTFOLIO_SLIPPAGE_VOLATILITY_SCALE", "").strip()),
]
for flag, value in optional_portfolio_arg_pairs:
    if value:
        portfolio_execution_cmd.extend([flag, value])
if env_true("NOTEBOOK15_PORTFOLIO_ENABLE_VOLATILITY_TARGETING"):
    portfolio_execution_cmd.append("--enable-volatility-targeting")
if env_true("NOTEBOOK15_RISK_ALLOW_SCALE_UP"):
    portfolio_execution_cmd.append("--risk-allow-scale-up")
if env_true("NOTEBOOK15_PORTFOLIO_EXECUTION_ENABLED"):
    portfolio_execution_cmd.append("--execution-enabled")
if env_true("NOTEBOOK15_PORTFOLIO_DISABLE_EXECUTION_MODEL"):
    portfolio_execution_cmd.append("--disable-execution-model")
if env_true("NOTEBOOK15_PORTFOLIO_STRICT"):
    portfolio_execution_cmd.append("--strict")

portfolio_archive_restore_cmd = ["stratlake-session-archive-restore-bootstrap", "--archive-root", STRATLAKE_ARCHIVE_DRIVE_PACK_DIR.as_posix(), "--target-root", STRATLAKE_ROOT.as_posix(), "--validate-before-restore", "--inspect-before-restore", "--overwrite-policy", os.environ.get("NOTEBOOK15_PORTFOLIO_RESTORE_OVERWRITE_POLICY", "overwrite_allowed")]
portfolio_input_reviewed = env_true("NOTEBOOK15_MARK_PORTFOLIO_INPUTS_USER_REVIEWED")
portfolio_execution_profile_ready = ALLOW_PORTFOLIO_EXECUTION and portfolio_input_reviewed and (PORTFOLIO_FROM_REGISTRY or bool(PORTFOLIO_RUN_IDS))
if ALLOW_PORTFOLIO_EXECUTION and not portfolio_input_reviewed:
    CAVEATS.append("Portfolio execution branch active but inputs were not explicitly reviewed; command remains a skipped preview.")
portfolio_archive_restore_result = run_command(portfolio_archive_restore_cmd, cwd=STRATLAKE_ROOT, allow_run=ALLOW_PORTFOLIO_ARCHIVE_RESTORE, label="portfolio archive restore")
portfolio_execution_result = run_command(portfolio_execution_cmd, cwd=STRATLAKE_ROOT, allow_run=portfolio_execution_profile_ready, label="native portfolio execution")
portfolio_artifact_root = PORTFOLIO_OUTPUT_DIR
portfolio_checkpoint = verify_checkpoint("portfolio_artifacts_available_or_restored", portfolio_artifact_root, required_when=ALLOW_PORTFOLIO_EXECUTION or ALLOW_PORTFOLIO_ARCHIVE_RESTORE, classification="required_portfolio_artifacts_for_review_and_archive", must_contain_any=["registry.jsonl", "*.json", "*.md", "*.parquet", "*.csv", "**/*.json", "**/*.md", "**/*.parquet", "**/*.csv"])
display_rows([{"surface": "portfolio_execution", "command": command_text(portfolio_execution_cmd), "profile_gate_active": ALLOW_PORTFOLIO_EXECUTION, "native_execution_confirmed": ALLOW_NATIVE_COMMAND_EXECUTION}, portfolio_checkpoint])


## 8. Portfolio Artifact Review and Archive Checkpoint

Portfolio artifact review is bounded and non-authoritative. Artifact presence does not prove execution completeness, approval, readiness, or investment suitability.

Fintech backup and restore both point at `<FINTECH_BACKUP_ROOT>/<backup_id>`. The notebook-level Fintech backup collision policy is `reuse_existing`, `new_id`, or `fail`; because `fintech-backup-data pack` has no native overwrite flag, `reuse_existing` validates an existing pack instead of overwriting it.

StratLake checkpoint and restore both use `<STRATLAKE_ARCHIVE_DRIVE_ROOT>/<archive_id>` for Drive-backed archive handoff. The native checkpoint keeps `--output-root` repository-relative as `artifacts/_derived/session_archives` and uses only valid include flags.


In [ ]:
expected_portfolio_artifacts = ["manifest.json", "metrics.json", "qa_summary.json", "portfolio_returns.csv", "portfolio_equity_curve.csv", "registry.jsonl"]
portfolio_artifact_rows = [{"expected_artifact": name, "root": portfolio_artifact_root.as_posix(), "found": bool((portfolio_artifact_root / name).exists() or any(portfolio_artifact_root.glob(f"**/{name}"))), "classification": "candidate_native_portfolio_artifact_display_only"} for name in expected_portfolio_artifacts]

fintech_backup_pack_exists = FINTECH_BACKUP_PACK_DIR.exists()
fintech_backup_pack_id = FINTECH_BACKUP_ID
if fintech_backup_pack_exists and FINTECH_BACKUP_COLLISION_POLICY == "new_id":
    fintech_backup_pack_id = f"{FINTECH_BACKUP_ID}-{int(time.time())}"
    FINTECH_BACKUP_PACK_DIR = FINTECH_BACKUP_ROOT / fintech_backup_pack_id
elif fintech_backup_pack_exists and FINTECH_BACKUP_COLLISION_POLICY == "fail":
    raise FileExistsError(f"Fintech backup pack already exists and collision policy is fail: {FINTECH_BACKUP_PACK_DIR.as_posix()}")

fintech_backup_cmd = ["fintech-backup-data", "pack", "--workspace-root", FINTECH_ROOT.as_posix(), "--source-dataset-root", DAILY_BARS_ROOT.as_posix(), "--backup-root", FINTECH_BACKUP_ROOT.as_posix(), "--backup-id", fintech_backup_pack_id, "--shard-size-mb", os.environ.get("NOTEBOOK15_FINTECH_BACKUP_SHARD_SIZE_MB", "64")]
fintech_backup_validate_cmd = ["fintech-backup-data", "validate", "--backup-pack-dir", FINTECH_BACKUP_PACK_DIR.as_posix()]
fintech_backup_operation_cmd = fintech_backup_validate_cmd if fintech_backup_pack_exists and FINTECH_BACKUP_COLLISION_POLICY == "reuse_existing" else fintech_backup_cmd
if not ALLOW_NATIVE_COMMAND_EXECUTION and fintech_backup_operation_cmd == fintech_backup_cmd:
    fintech_backup_operation_cmd.append("--dry-run")

STRATLAKE_ARCHIVE_OUTPUT_ROOT = os.environ.get("NOTEBOOK15_STRATLAKE_ARCHIVE_OUTPUT_ROOT", "artifacts/_derived/session_archives").strip() or "artifacts/_derived/session_archives"
if Path(STRATLAKE_ARCHIVE_OUTPUT_ROOT).is_absolute():
    raise ValueError("NOTEBOOK15_STRATLAKE_ARCHIVE_OUTPUT_ROOT must be repository-relative.")
STRATLAKE_LOCAL_ARCHIVE_PACK_DIR = STRATLAKE_ROOT / STRATLAKE_ARCHIVE_OUTPUT_ROOT / STRATLAKE_ARCHIVE_ID
STRATLAKE_ARCHIVE_COLLISION_POLICY = os.environ.get("NOTEBOOK15_STRATLAKE_ARCHIVE_COLLISION_POLICY", "overwrite_allowed").strip() or "overwrite_allowed"
STRATLAKE_ARCHIVE_COPY_POLICY = os.environ.get("NOTEBOOK15_STRATLAKE_ARCHIVE_COPY_POLICY", "overwrite_allowed").strip() or "overwrite_allowed"
if STRATLAKE_ARCHIVE_COLLISION_POLICY not in {"fail_if_exists", "overwrite_allowed"}:
    raise ValueError("NOTEBOOK15_STRATLAKE_ARCHIVE_COLLISION_POLICY must be fail_if_exists or overwrite_allowed")
if STRATLAKE_ARCHIVE_COPY_POLICY not in {"fail_if_exists", "overwrite_allowed", "skip_existing"}:
    raise ValueError("NOTEBOOK15_STRATLAKE_ARCHIVE_COPY_POLICY must be fail_if_exists, overwrite_allowed, or skip_existing")
stratlake_checkpoint_cmd = ["stratlake-session-archive-bootstrap", "--root", STRATLAKE_ROOT.as_posix(), "--archive-id", STRATLAKE_ARCHIVE_ID, "--output-root", STRATLAKE_ARCHIVE_OUTPUT_ROOT, "--include-features", "--include-artifacts", "--include-configs", "--archive-collision-policy", STRATLAKE_ARCHIVE_COLLISION_POLICY, "--copy-policy", STRATLAKE_ARCHIVE_COPY_POLICY, "--validate-after-copy", "--inspect-after-copy"]
if env_true("NOTEBOOK15_INCLUDE_DUCKDB_SNAPSHOT"):
    stratlake_checkpoint_cmd.append("--include-duckdb-snapshot")

archive_market_checkpoint = verify_checkpoint("archive_market_data_checkpoint", DAILY_BARS_ROOT, required_when=ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT and (ALLOW_FINTECH_INGESTION or ALLOW_FINTECH_ARCHIVE_RESTORE or ALLOW_FEATURE_GENERATION), classification="archive_prerequisite_market_data", must_contain_any=["*.parquet", "*.csv", "**/*.parquet", "**/*.csv"])
archive_feature_checkpoint = verify_checkpoint("archive_feature_checkpoint", FEATURES_DAILY_ROOT, required_when=ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT and (ALLOW_FEATURE_GENERATION or ALLOW_FEATURE_ARCHIVE_RESTORE or ALLOW_STRATEGY_EXECUTION), classification="archive_prerequisite_feature_data", must_contain_any=["*.parquet", "*.csv", "**/*.parquet", "**/*.csv"])
archive_strategy_checkpoint = verify_checkpoint("archive_strategy_checkpoint", strategy_artifact_review_root, required_when=ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT and (ALLOW_STRATEGY_EXECUTION or ALLOW_STRATEGY_ARTIFACT_RESTORE or ALLOW_PORTFOLIO_EXECUTION), classification="archive_prerequisite_strategy_artifacts", must_contain_any=["*.json", "*.parquet", "*.csv", "**/*.json", "**/*.parquet", "**/*.csv"])
archive_portfolio_checkpoint = verify_checkpoint("archive_portfolio_checkpoint", portfolio_artifact_root, required_when=ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT and (ALLOW_PORTFOLIO_EXECUTION or ALLOW_PORTFOLIO_ARCHIVE_RESTORE), classification="archive_prerequisite_portfolio_artifacts", must_contain_any=["*.json", "*.md", "*.parquet", "*.csv", "**/*.json", "**/*.md", "**/*.parquet", "**/*.csv"])
fintech_backup_result = run_command(fintech_backup_operation_cmd, allow_run=ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT, label="fintech archive backup or existing pack validation")
stratlake_checkpoint_result = run_command(stratlake_checkpoint_cmd, cwd=STRATLAKE_ROOT, allow_run=ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT, label="stratlake full workflow archive checkpoint")
if ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT and ALLOW_NATIVE_COMMAND_EXECUTION and STRATLAKE_LOCAL_ARCHIVE_PACK_DIR.exists():
    if STRATLAKE_ARCHIVE_DRIVE_PACK_DIR.exists() and STRATLAKE_ARCHIVE_COPY_POLICY == "fail_if_exists":
        raise FileExistsError(f"StratLake Drive archive pack already exists: {STRATLAKE_ARCHIVE_DRIVE_PACK_DIR.as_posix()}")
    if STRATLAKE_ARCHIVE_DRIVE_PACK_DIR.exists() and STRATLAKE_ARCHIVE_COPY_POLICY == "overwrite_allowed":
        shutil.rmtree(STRATLAKE_ARCHIVE_DRIVE_PACK_DIR)
    if not STRATLAKE_ARCHIVE_DRIVE_PACK_DIR.exists():
        shutil.copytree(STRATLAKE_LOCAL_ARCHIVE_PACK_DIR, STRATLAKE_ARCHIVE_DRIVE_PACK_DIR)
display_rows(portfolio_artifact_rows + [archive_market_checkpoint, archive_feature_checkpoint, archive_strategy_checkpoint, archive_portfolio_checkpoint])


## 9. Final Review Summary

The final summary records command previews, checkpoints, caveats, and non-claims.


In [ ]:
NON_CLAIMS = ["investment_recommendation", "strategy_approval", "positive_alpha", "statistical_significance", "promotion_readiness", "governance_readiness", "deployment_readiness", "production_readiness", "live_trading_suitability", "source_runtime_equivalence", "portfolio_performance_quality", "authoritative_performance_reporting", "native_artifact_completeness_without_native_verification"]
command_records_executed = sum(1 for row in COMMAND_RESULTS if not row.get("skipped"))
command_records_skipped = sum(1 for row in COMMAND_RESULTS if row.get("skipped"))
command_records_skipped_previews = sum(1 for row in COMMAND_RESULTS if row.get("skipped_preview"))
required_checkpoints = [row for row in CHECKPOINT_RESULTS if row.get("required_by_profile")]
missing_required_checkpoints = [row for row in required_checkpoints if not row.get("satisfied")]
portfolio_case_study = {"case_study_name": "notebook15_portfolio_workflow_review", "native_surface": "stratlake-run-portfolio", "portfolio_specific_command_confirmed": True, "portfolio_config": PORTFOLIO_CONFIG_PATH.as_posix(), "portfolio_name": PORTFOLIO_NAME, "from_registry": PORTFOLIO_FROM_REGISTRY, "timeframe": PORTFOLIO_TIMEFRAME}
archive_handoff = {"fintech_backup_pack_dir": FINTECH_BACKUP_PACK_DIR.as_posix(), "fintech_restore_pack_dir": FINTECH_BACKUP_PACK_DIR.as_posix(), "stratlake_archive_drive_pack_dir": STRATLAKE_ARCHIVE_DRIVE_PACK_DIR.as_posix(), "stratlake_restore_archive_root": STRATLAKE_ARCHIVE_DRIVE_PACK_DIR.as_posix(), "stratlake_archive_output_root": STRATLAKE_ARCHIVE_OUTPUT_ROOT}
final_review_summary = {
    "notebook": "Notebook 15 - Portfolio Workflow Review and Case Study",
    "selected_profile": NOTEBOOK15_PROFILE,
    "enabled_profile_gates": sorted(actual_enabled_gates),
    "native_command_execution_confirmed": ALLOW_NATIVE_COMMAND_EXECUTION,
    "workspace_initialization_enabled": ALLOW_WORKSPACE_INITIALIZATION,
    "source_safe_committed_default": NOTEBOOK15_PROFILE == "portfolio_preview",
    "command_records_total": len(COMMAND_RESULTS),
    "command_records_executed": command_records_executed,
    "command_records_skipped": command_records_skipped,
    "command_records_skipped_previews": command_records_skipped_previews,
    "checkpoint_records_total": len(CHECKPOINT_RESULTS),
    "required_checkpoints_total": len(required_checkpoints),
    "required_checkpoints_missing": len(missing_required_checkpoints),
    "missing_required_checkpoint_labels": [row["checkpoint"] for row in missing_required_checkpoints],
    "selected_strategies": SELECTED_STRATEGIES,
    "portfolio_case_study": portfolio_case_study,
    "portfolio_command": command_text(portfolio_execution_cmd),
    "archive_handoff": archive_handoff,
    "caveat_count": len(CAVEATS),
    "caveats": CAVEATS,
    "non_claims": NON_CLAIMS,
    "final_stance": "notebook_15_native_portfolio_workflow_source_safe_review_ready",
}
final_review_summary
